# Fine-tune the NY Tree ID model on a free GPU

This notebook trains the **more accurate** version of the tree model by fine-tuning the whole
network (the model shipped first was trained on a CPU with the backbone frozen).

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. Total time is about 1–1.5 hours (≈15 min downloading photos, the rest training).
3. At the end you download `nys_tree_model.zip`. Put its three files in
   `android/app/src/main/assets/` and rebuild the app.

In [ ]:
# 1. Get the code. If the repository is private, paste a GitHub token with read access when asked.
import getpass, os, subprocess
REPO = "Chrysalis-Technologies/Chestnut_Hills"
BRANCH = "claude/offline-android-tree-id-42p056"
if not os.path.exists("Chestnut_Hills"):
    token = getpass.getpass("GitHub token (leave empty for a public repo): ").strip()
    auth = f"{token}@" if token else ""
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    f"https://{auth}github.com/{REPO}.git", "Chestnut_Hills"], check=True)
%cd Chestnut_Hills
!nvidia-smi -L

In [ ]:
# 2. Python packages (Colab already has TensorFlow; this just makes sure the rest is there).
!pip -q install pillow
import tensorflow as tf; print("TensorFlow", tf.__version__, "GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
# 3. Download the ~100k training photos listed in the manifest (openly licensed iNaturalist photos).
#    Safe to re-run: already-downloaded photos are skipped.
!python training/download_images.py --manifest training/data/manifest.csv.gz --out-dir /content/images --workers 48

In [ ]:
# 4. Fine-tune. Try --backbone efficientnetv2b2 for a bigger, slower, usually more accurate model.
!python training/train.py --mode finetune --images /content/images \
    --out-dir /content/model --work-dir /content/work --batch 64 --epochs 15

In [ ]:
# 5. Results on photos the model never saw during training.
from IPython.display import Markdown, display
display(Markdown(open("/content/model/eval_report.md").read()))

In [ ]:
# 6. Download the model files for the app.
!cd /content/model && zip -q /content/nys_tree_model.zip tree_model.tflite labels.txt model_info.json eval_report.md
from google.colab import files
files.download("/content/nys_tree_model.zip")